In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower

# 1. Load the Cookie Cats dataset
df = pd.read_csv("../data/cookie_cats.csv")
print("Dataset shape:", df.shape)
display(df.head())

# Check Sample Ratio Mismatch (SRM) between Control (gate_30) and Treatment (gate_40)
group_counts = df["version"].value_counts()
print("\nGroup Counts:\n", group_counts)
chi2_srm, p_srm = stats.chisquare(group_counts)
print(f"SRM Check p-value: {p_srm:.4f} ({'Pass - Balanced split' if p_srm > 0.01 else 'Fail - Imbalanced'})")


Dataset shape: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True



Group Counts:
 version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
SRM Check p-value: 0.0086 (Fail - Imbalanced)


In [ ]:
# 2. A/B Test Analysis: 7-Day Retention (gate_30 vs gate_40)
# H0: No difference in 7-day retention between gate_30 (Control) and gate_40 (Treatment)
# H1: Moving the gate to level 40 changes 7-day retention

control = df[df["version"] == "gate_30"]["retention_7"]
treatment = df[df["version"] == "gate_40"]["retention_7"]

n_ctrl, n_treat = len(control), len(treatment)
conv_ctrl, conv_treat = control.sum(), treatment.sum()
rate_ctrl, rate_treat = control.mean(), treatment.mean()
diff = rate_treat - rate_ctrl

# Two-Proportion Z-Test
z_stat, p_val = proportions_ztest([conv_treat, conv_ctrl], [n_treat, n_ctrl], alternative="two-sided")

# 95% Confidence Interval for the difference in proportions
se_diff = np.sqrt((rate_ctrl * (1 - rate_ctrl) / n_ctrl) + (rate_treat * (1 - rate_treat) / n_treat))
ci_low, ci_high = diff - 1.96 * se_diff, diff + 1.96 * se_diff

# Statistical Power
effect_size = abs(diff) / np.sqrt(((rate_ctrl * (1 - rate_ctrl)) + (rate_treat * (1 - rate_treat))) / 2)
power = NormalIndPower().solve_power(effect_size=effect_size, nobs1=n_ctrl, alpha=0.05, ratio=n_treat/n_ctrl)

print(f"Control (gate_30) 7-Day Retention:   {rate_ctrl:.4%}")
print(f"Treatment (gate_40) 7-Day Retention: {rate_treat:.4%}")
print(f"Absolute Lift (Treatment - Control): {diff:+.4%}")
print(f"Z-Statistic:                         {z_stat:.4f}")
print(f"p-value:                             {p_val:.4f}")
print(f"95% Confidence Interval:             [{ci_low:+.4%}, {ci_high:+.4%}]")
print(f"Statistical Power:                   {power:.2%}")
print(f"Conclusion: {'Statistically Significant (Reject H0)' if p_val < 0.05 else 'Not Significant (Fail to Reject H0)'}")

# 3. Bootstrap Distribution Visualization
np.random.seed(42)
boot_diffs = [
    df.sample(frac=1, replace=True).groupby("version")["retention_7"].mean().pipe(lambda s: s["gate_40"] - s["gate_30"])
    for _ in range(1000)
]

plt.figure(figsize=(8, 4))
sns.histplot(boot_diffs, kde=True, color="#2b5c8f")
plt.axvline(0, color="red", linestyle="--", label="No Difference (0%)")
plt.axvline(diff, color="green", linestyle="-", label=f"Observed Diff ({diff:+.2%})")
plt.title("Bootstrap Distribution of 7-Day Retention Difference (gate_40 - gate_30)")
plt.xlabel("Difference in 7-Day Retention Rate")
plt.legend()
plt.tight_layout()
plt.show()